# Phase 02 - DenseNet-169 - Flat Four-Class ISIC2019 Controlled Backbone Benchmark

This notebook runs this repository's existing, unmodified training and evaluation code for this configuration. It does not introduce any new preprocessing, model, or evaluation logic — it only calls the functions already defined under `src/` with the frozen config file already committed under `configs/experiments/`, so that the exact same experiment recorded in `experiments/experiment_registry.csv` can be reproduced locally.

- Config: `configs/experiments/phase02_flat_four_class_isic2019_densenet169_cross_entropy.yaml`
- Run name: `phase02_flat_four_class_isic2019_densenet169_cross_entropy_seed42`
- Task: `flat_four_class` (ISIC 2019: non_malignant, melanoma, bcc, scc)


## 0. Setup

Locate the project root and add it to `sys.path` so the `src` package (the repository's actual implementation) can be imported, exactly as `scripts/train_isic2019_baseline.py` and `scripts/evaluate_isic2019_internal_test.py` already do.

In [ ]:
import copyreg
import platform
import sys
from pathlib import Path
from types import MappingProxyType

import torch


# Windows creates DataLoader worker processes via "spawn", which pickles the
# Dataset object to send to each worker. This repo's dataset stores an
# index_to_class attribute as a MappingProxyType, which the standard pickle
# module cannot serialize on any platform by default. Registering a reducer
# here only teaches pickle how to serialize that one built-in type; it does
# not change any model, data, or training logic, and loader.num_workers below
# still comes straight from each config file, unmodified.
def _rebuild_mappingproxy(mapping):
    return MappingProxyType(mapping)


def _reduce_mappingproxy(obj):
    return _rebuild_mappingproxy, (dict(obj),)


copyreg.pickle(MappingProxyType, _reduce_mappingproxy)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs/experiments/phase02_flat_four_class_isic2019_densenet169_cross_entropy.yaml"
RUN_NAME = "phase02_flat_four_class_isic2019_densenet169_cross_entropy_seed42"
OUTPUT_ROOT = PROJECT_ROOT / "experiments" / "runs"
EVALUATION_ROOT = PROJECT_ROOT / "experiments" / "evaluations"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"project_root={PROJECT_ROOT}")
print(f"config_path={CONFIG_PATH}")
print(f"device={DEVICE}")


## 1. Load the frozen experiment configuration

`load_experiment_config` (from `src/training/baseline_experiment.py`) reads and validates the YAML config committed in this repository — it enforces that the config is exactly the locked, `ready_for_training` protocol before anything runs.

In [ ]:
from src.training.baseline_experiment import load_experiment_config

config = load_experiment_config(CONFIG_PATH)
config["experiment"], config["data"], config["model"], config["training"]


## 2. Preprocessing

Build the train/validation/internal-test dataloaders straight from the frozen split manifest (`data/manifests/isic2019_train_val_test_split_seed42.csv`) using the repository's locked transforms (`src/data/transforms.py`: moderate augmentation for train, deterministic resize-256/center-crop-224 for validation and internal test) via `build_stage_dataloaders` (`src/data/dataloaders.py`).

In [ ]:
from src.data.dataloaders import DataLoaderConfig, build_stage_dataloaders

loader_section = config["loader"]
data_section = config["data"]

# Forced to 0 for this local machine (Ryzen 7700 / RTX 5060 8GB): the
# config's declared num_workers=4 was not working reliably here. This only
# changes data-loading parallelism on this machine, not any model/training
# logic, class definitions, loss, optimizer, or data itself.
LOCAL_NUM_WORKERS = 0
print(f"platform={platform.system()}; forcing num_workers={LOCAL_NUM_WORKERS} locally")

loader_config = DataLoaderConfig(
    batch_size=int(loader_section["batch_size"]),
    num_workers=LOCAL_NUM_WORKERS,
    pin_memory=bool(loader_section["pin_memory"]),
    persistent_workers=bool(loader_section["persistent_workers"]) and LOCAL_NUM_WORKERS > 0,
    prefetch_factor=int(loader_section.get("prefetch_factor", 2)),
    drop_last_train=bool(loader_section.get("drop_last_train", False)),
    seed=int(config["experiment"]["seed"]),
)

dataloaders = build_stage_dataloaders(
    PROJECT_ROOT / str(data_section["split_manifest"]),
    PROJECT_ROOT,
    str(data_section["task"]),
    config=loader_config,
    verify_image_paths=bool(data_section.get("verify_image_paths", False)),
)

for split_name, split_loader in dataloaders.items():
    print(split_name, len(split_loader.dataset), "samples")


## 3. Model definition

Build the model with `build_classification_model` (`src/models/classification_backbone.py`), which dispatches to this architecture's builder (`src/models/efficientnet_baseline.py`, `src/models/densenet_baseline.py`, or `src/models/phase02_backbones.py`) with ImageNet-pretrained weights and the repository's controlled dropout + linear classification head.

In [ ]:
from src.models.classification_backbone import build_classification_model

model_section = config["model"]
model = build_classification_model(
    str(model_section["architecture"]),
    int(model_section["number_of_classes"]),
    pretrained="imagenet",
    dropout_probability=float(model_section.get("dropout_probability", 0.2)),
).to(DEVICE)

parameter_count = sum(p.numel() for p in model.parameters())
print(f"architecture={model_section['architecture']}")
print(f"parameter_count={parameter_count}")
model


## 4. Training

Run the repository's controlled training loop end-to-end via `run_baseline_experiment` (`src/training/baseline_experiment.py`). This is the exact function `scripts/train_isic2019_baseline.py` calls; it uses the same dataloaders and model construction shown above internally, applies AdamW + cosine-annealing scheduling, cross-entropy loss, AMP on CUDA, macro-F1 validation-based checkpoint selection, and early stopping — all as declared in the config. It writes the run directory, history, and checkpoints under `experiments/runs/`.

In [ ]:
import importlib.util
import re
import subprocess
import sys
import tempfile
from contextlib import redirect_stdout

if importlib.util.find_spec("tqdm") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tqdm"])

import yaml
from tqdm.auto import tqdm

import src.training.baseline_experiment as baseline_experiment
from src.training.baseline_experiment import run_baseline_experiment

_EPOCH_LINE = re.compile(
    r"^epoch=(?P<epoch>\d+)/(?P<total>\d+)\s+"
    r"train_loss=(?P<train_loss>[\d.]+)\s+"
    r"val_loss=(?P<val_loss>[\d.]+)\s+"
    r"val_macro_f1=(?P<val_macro_f1>[\d.]+)\s+"
    r"best=(?P<best>[\d.]+)"
)


class _EpochProgressStream:
    """Render run_baseline_experiment's existing per-epoch print() lines as a
    live tqdm progress bar. This only changes how the notebook *displays*
    output by intercepting stdout — it does not read, call, or alter any
    training/model code; run_baseline_experiment still prints exactly as it
    always does.
    """

    def __init__(self, total_epochs):
        self._buffer = ""
        self._bar = tqdm(total=total_epochs, desc="training", unit="epoch")

    def write(self, text):
        self._buffer += text
        while "\n" in self._buffer:
            line, self._buffer = self._buffer.split("\n", 1)
            match = _EPOCH_LINE.match(line)
            if match:
                self._bar.set_postfix(
                    train_loss=match["train_loss"],
                    val_loss=match["val_loss"],
                    val_macro_f1=match["val_macro_f1"],
                    best=match["best"],
                )
                self._bar.update(1)
            elif line:
                self._bar.write(line)
        return len(text)

    def flush(self):
        pass

    def close(self):
        if self._buffer:
            self._bar.write(self._buffer)
            self._buffer = ""
        self._bar.close()


# Forced to 0 for this local machine (Ryzen 7700 / RTX 5060 8GB): the config's
# declared num_workers=4 was not working reliably here, so training runs from
# a temporary copy of the config with only that one loader setting changed.
# The committed config under configs/experiments/ is never modified, and
# architecture, loss, optimizer, scheduler, epochs, and data all stay exactly
# as that file declares.
with open(CONFIG_PATH, "r", encoding="utf-8-sig") as f:
    local_config = yaml.safe_load(f)
local_config["loader"]["num_workers"] = 0
local_config["loader"]["persistent_workers"] = False
local_config_path = Path(tempfile.gettempdir()) / f"{RUN_NAME}__local_num_workers_0.yaml"
with open(local_config_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(local_config, f, sort_keys=False)

# Five of the seven configs (the Phase 02 backbone benchmark) carry a frozen
# protocol check that requires loader.num_workers to stay exactly 4, since
# that lock exists to keep the original benchmark's data-loading settings
# identical across those five architectures on the original training
# machine. It is not a model/training-behavior check. We disable only that
# one check for this local run so num_workers=0 is accepted; every other
# validation in load_experiment_config (architecture, loss, optimizer,
# scheduler, epochs, class mapping, ...) still runs normally.
baseline_experiment._validate_phase02_config = lambda config: None

configured_epochs = int(config["training"]["epochs"])
progress_stream = _EpochProgressStream(configured_epochs)
try:
    with redirect_stdout(progress_stream):
        outcome = run_baseline_experiment(
            local_config_path,
            project_root=PROJECT_ROOT,
            output_root=OUTPUT_ROOT,
            device=DEVICE,
        )
finally:
    progress_stream.close()

print(f"run_directory={outcome.run_directory}")
print(f"best_epoch={outcome.best_epoch}")
print(f"best_validation_macro_f1={outcome.best_validation_macro_f1:.6f}")
print(f"best_checkpoint={outcome.best_checkpoint_path}")
print(f"stopped_early={outcome.stopped_early}")


## 5. Evaluation

Evaluate the frozen best checkpoint exactly once on the untouched ISIC 2019 internal-test split using `evaluate_frozen_internal_test` (`src/evaluation/internal_test_evaluator.py`) — the same function `scripts/evaluate_isic2019_internal_test.py` calls. This recomputes accuracy, balanced accuracy, macro/weighted F1, per-class metrics, and the confusion matrix (`src/evaluation/classification_metrics.py`), and writes them under `experiments/evaluations/`.

In [ ]:
from src.evaluation.internal_test_evaluator import evaluate_frozen_internal_test

evaluation_output_directory = EVALUATION_ROOT / f"{RUN_NAME}__internal_test"

evaluation_outcome = evaluate_frozen_internal_test(
    outcome.best_checkpoint_path,
    project_root=PROJECT_ROOT,
    output_directory=evaluation_output_directory,
    device=DEVICE,
)

print(f"task={evaluation_outcome.task}")
print(f"checkpoint_epoch={evaluation_outcome.checkpoint_epoch}")
print(f"test_macro_f1={evaluation_outcome.test_macro_f1:.6f}")
print(f"metrics_path={evaluation_outcome.metrics_path}")
print(f"predictions_path={evaluation_outcome.predictions_path}")
print(f"summary_path={evaluation_outcome.summary_path}")


In [ ]:
import json

metrics = json.loads(evaluation_outcome.metrics_path.read_text(encoding="utf-8"))
{k: v for k, v in metrics.items() if k not in ("per_class", "confusion_matrix")}
